In [1]:
from google.colab import drive; drive.mount('/content/drive')
import os
print(os.listdir('/content/drive/MyDrive/data/names'))


Mounted at /content/drive
['Czech.txt', 'German.txt', 'Japanese.txt', 'Chinese.txt', 'English.txt', 'Irish.txt', 'Greek.txt', 'Italian.txt', 'Vietnamese.txt', 'Spanish.txt', 'Arabic.txt', 'Portuguese.txt', 'Scottish.txt', 'Russian.txt', 'Dutch.txt', 'Polish.txt', 'Korean.txt', 'French.txt']


In [2]:
from io import open 
import glob 
import os 
import unicodedata
import string
import torch 
import torch.nn as nn 
import torch.nn.functional as F 


### imports

In [3]:
all_lettrs = string.ascii_letters + " .,;'-"
n_lettrs = len(all_lettrs) + 1 

def findFiles(path): return glob.blob(path)

def unicodeToAscii(s): 
    return ''.join(
        c for c in unicodedata.normalize('NFD', s) 
        if unicodedata.category(c) != 'Mn'
        and c in all_lettrs
    )

def readLines(filename): 
    with open (filename, encoding = 'utf-8') as some_file: 
        return [unicodeToAscii(line.strip()) for line in some_file]
    
    category_lines = {} 
    all_cats = [] 

    for filename in findFiles('data/names/*.txt'):
        category = os.path.splitext(os.path.basename(filename))[0]
        all_cats.append(category) 
        lines = readLines(filename)
        category_lines[category] = lines

    n_cats = len(all_cats) 

    if n_cats == 0: 
        raise RuntimeError('Data not found') 

    print('#categories:', n_cats, all_cats)
    print(unicodeToAscii("O'Néàl"))

In [4]:
import torch 
import torch.nn as nn

class RNN(nn.Module): 
    def __init__(self, input_size, hidden_size, output_size):
        super(RNN, self).__init__() 

        self.hidden_size = hidden_size
        self.i2h = nn.Linear(n_cats + input_size + hidden_size, hidden_size)
        self.i20 = nn.Linear(n_cats, input_size, hidden_size, output_size)
        self.o20 = nn.Linear(hidden_size + output_size, output_size)
        self.dropout = nn.DropOut(0.1)  
        self.softmax = nn.LogSoftmax(dim = 1) 

    def forward(self, category, input, hidden): 
        input_combined = torch.cat((category, input, hidden), 1)
        hidden = self.i2h(input_combined) 
        output = self.i2o(input_combined) 
        output_combined = torch.cat((hidden, output), 1)
        output = self.o2o(output_combined) 
        output = self.dropout(output)
        output = self.softmax(output) 
        return output, hidden 

    def initHidden(self): 
        return torch.zeros(1, self.hidden_size)
        


In [5]:
import random 

def randomChoice(l): 
    return l[random.randint(0, len(l) - 1)]

def randomTrainingPair():
    category = randomChoice(all_cats)
    line = randomChoice(cats_line[category])
    return category, line





In [6]:
def categoryTensor(category):
    li = all_categories.index(category) 
    tensor = torch.zeroes(1, n_cats)
    tensor[0][li] = 1
    return tensor 

def inputTensor(line): 
    tensor = torch.zeros(len(line), 1, n_lettrs)
    for li in range(len(line)): 
        letter = line[li] 
        tensor[li][0][all_lettrs.find(letter)] = 1
    
    return tensor 

def targetTensor(line): 
    letter_indexes = [all_lettrs.find(line[li]) for li in range(1, len(line))] 
    letter_indexes.append(n_lettrs - 1)
    return torch.LongTensor(letter_indexes) 



In [7]:
def randomTrainingExample(): 
    category, line = randomTrainingPair() 
    category_tensor = categoryTensor(category) 
    input_line_tensor = inputTensor(line)
    target_line_tensor = targetTensor(line) 
    return category_tensor, input_line_tensor, target_line_tensor



criterion = nn.NLLLoss() 
learning_rate = 0.0005 

def train(category_tensor, input_line_tensor, target_line_tensor): 
    target_line_tensor.unsqueeze_(-1)
    hidden = rnn.initHidden() 

    rnn.zero_grad() 

    loss = torch.Tensor([0]) 

    for i in range(input_line_tensor.sizse(0)): 
        output, hidden = rnn(category_tensor, input_line_tensor[i], hidden)
        li = criterion(output, target_line_tensor[i]) 
        loss += 1

    loss.backward() 

    for p in rnn.parameters(): 
        p.data.add_(p.grad.data, alpha=-learning_rate)
    
    return output, loss.item() / input_line_tensor.size(0)




In [8]:
import time
import math 

def timeSince(since): 
    now = time.time() 
    s = now - since
    m = math.floor(s / 60) 
    s -= m * 60 

    return '%dm %ds' % (m, s) 


In [ ]:
rnn = RNN(n_lettrs, 128, n_lettrs)

n_iters = 100000
print_evr = 5000
plot_evry = 550 
all_loss = [] 
tot_loss = 0 

start = time.time() 

for iter in range(1, n_iters + 1): 
    output, loss = train(*randomTrainingExample()) 
    tot_loss += loss

    if iter % print_evr == 0: 
        print(",_")
    if iter % plot_evry == 0: 
        all_loss.append(tot_loss / plot_evry) 
        tot_loss = 0 